# Workshop: Streaming & Auto Loader

> *"Set up Auto Loader for streaming JSON ingestion into the Bronze layer with exactly-once guarantees."*

## Setup

In [0]:
%run ../../setup/00_setup

In [0]:
# Prepare landing zone and checkpoint paths
landing_path = f"{DATASET_PATH}/orders/stream"
checkpoint_path = f"{DATASET_PATH}/tmp/{CATALOG}/lab05/checkpoint"
schema_path = f"{DATASET_PATH}/tmp/{CATALOG}/lab05/schema"
target_table = f"{CATALOG}.{BRONZE_SCHEMA}.orders_stream"

# Clean up from previous runs
spark.sql(f"DROP TABLE IF EXISTS {target_table}")
dbutils.fs.rm(checkpoint_path, True)
dbutils.fs.rm(schema_path, True)

# Ensure customers table exists (needed for Task 8: Stream-Static Join)
customers_path = f"{DATASET_PATH}/customers/customers.csv"
if not spark.catalog.tableExists(f"{CATALOG}.{BRONZE_SCHEMA}.customers"):
    spark.read.format("csv").option("header", True).option("inferSchema", True) \
        .load(customers_path).write.mode("overwrite").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.customers")
    print(f"Created: {CATALOG}.{BRONZE_SCHEMA}.customers")

print(f"Landing path: {landing_path}")
print(f"Target table: {target_table}")
print(f"Files available: {[f.name for f in dbutils.fs.ls(landing_path)]}")

## Task 1: COPY INTO (Batch Ingestion)

Use `COPY INTO` to load the first file from the landing zone.

In [0]:
# First create the target table (provided)
spark.sql(f"""
    CREATE OR REPLACE TABLE {target_table}
    (order_id STRING, customer_id STRING, product_id STRING, store_id STRING,
     order_datetime STRING, quantity BIGINT, unit_price DOUBLE,
     discount_percent BIGINT, total_amount DOUBLE, payment_method STRING)
""")

# TODO: Write a COPY INTO statement to load JSON files from landing_path
# Use FILEFORMAT = JSON
# Add FORMAT_OPTIONS ('mergeSchema' => 'true')
spark.sql(f"""
    -- YOUR CODE HERE
""")

count_after_copy = spark.table(target_table).count()
print(f"Rows after COPY INTO: {count_after_copy}")

In [0]:
# -- Validation --
assert count_after_copy > 0, "COPY INTO should have loaded data"
print(f"Task 1 OK: {count_after_copy} rows loaded via COPY INTO")

## Task 2: Verify COPY INTO Idempotency

Run COPY INTO again on the same files. How many new rows are loaded?

In [0]:
# TODO: Re-run the same COPY INTO
spark.sql(f"""
    -- YOUR CODE HERE
""")

count_after_rerun = spark.table(target_table).count()
print(f"Rows after re-run: {count_after_rerun} (was {count_after_copy})")
print(f"New rows loaded: {count_after_rerun - count_after_copy}")

In [0]:
# -- Validation --
assert count_after_rerun == count_after_copy, "COPY INTO should be idempotent - no new rows!"
print("Task 2 OK: COPY INTO is idempotent. 0 new rows on re-run.")

## Task 3: Auto Loader - Configure Stream

Set up Auto Loader (`cloudFiles`) to read JSON files from the landing zone.

Key options:
- `cloudFiles.format` = json
- `cloudFiles.schemaLocation` = path for inferred schema
- `cloudFiles.inferColumnTypes` = true

In [0]:
# Reset target for Auto Loader test
al_target = f"{CATALOG}.{BRONZE_SCHEMA}.orders_autoloader"
spark.sql(f"DROP TABLE IF EXISTS {al_target}")
dbutils.fs.rm(checkpoint_path, True)
dbutils.fs.rm(schema_path, True)

In [0]:
# TODO: Configure Auto Loader readStream
# - Use .format("cloudFiles") for Auto Loader
# - Set cloudFiles.format to "json"
# - Set cloudFiles.schemaLocation to schema_path
# - Enable cloudFiles.inferColumnTypes
df_stream = (
    # YOUR CODE HERE
    ...
)

In [0]:
# -- Validation --
assert df_stream.isStreaming, "Should be a streaming DataFrame"
print(f"Task 3 OK: Streaming DataFrame configured with schema: {df_stream.schema.fieldNames()}")

## Task 4: Write Stream with trigger(availableNow=True)

Write the stream to a Delta table using `trigger(availableNow=True)`.

This processes all available files and stops automatically.

In [0]:
# TODO: Write df_stream to al_target as a Delta table
# Use outputMode "append", checkpointLocation from checkpoint_path
# Use trigger(availableNow=True) — processes all pending files and stops
query = (
    # YOUR CODE HERE
    ...
)

query.awaitTermination()
print(f"Stream completed. Rows loaded: {spark.table(al_target).count()}")

In [0]:
# -- Validation --
al_count = spark.table(al_target).count()
assert al_count > 0, "Auto Loader should have loaded data"
print(f"Task 4 OK: {al_count} rows loaded via Auto Loader")

## Task 5: Incremental Processing

Re-run the stream. Since no new files arrived, 0 new rows should be processed.

This proves the checkpoint tracks processed files.

In [0]:
# Re-run the stream (same checkpoint = incremental)
df_stream2 = (
    # YOUR CODE HERE — same readStream config as Task 3
    ...
)

query2 = (
    # YOUR CODE HERE — same writeStream config as Task 4
    ...
)

query2.awaitTermination()
al_count2 = spark.table(al_target).count()
print(f"Rows after re-run: {al_count2} (was {al_count})")
print(f"New rows: {al_count2 - al_count}")

In [0]:
# -- Validation --
assert al_count2 == al_count, f"Should be 0 new rows, but got {al_count2 - al_count}"
print("Task 5 OK: Incremental processing verified. 0 new rows on re-run (checkpoint works!)")

## Task 6: Metadata Columns

Add metadata columns to streaming data: `_processing_time` (processing timestamp) and `_source_file` (source file path from `_metadata`).

These columns are essential in production pipelines for debugging and auditing.

**TODO:** Fill in `current_timestamp()` and `_metadata.file_path`.

In [0]:
from pyspark.sql.functions import current_timestamp, col

metadata_target = f"{CATALOG}.{BRONZE_SCHEMA}.orders_with_metadata"
metadata_checkpoint = f"/tmp/{CATALOG}/lab05/checkpoint_metadata"
metadata_schema = f"/tmp/{CATALOG}/lab05/schema_metadata"

spark.sql(f"DROP TABLE IF EXISTS {metadata_target}")
dbutils.fs.rm(metadata_checkpoint, True)
dbutils.fs.rm(metadata_schema, True)

# TODO: Configure Auto Loader and add two metadata columns:
# - _processing_time: use current_timestamp()
# - _source_file: use col("_metadata.file_path")
df_with_metadata = (
    # YOUR CODE HERE
    ...
)

(
    df_with_metadata
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", metadata_checkpoint)
    .trigger(availableNow=True)
    .toTable(metadata_target)
    .awaitTermination()
)

print(f"Written to: {metadata_target}")
display(spark.table(metadata_target).limit(5))

In [0]:
# -- Validation --
meta_cols = spark.table(metadata_target).columns
assert "_processing_time" in meta_cols, "Missing '_processing_time' — did you use current_timestamp()?"
assert "_source_file" in meta_cols, "Missing '_source_file' — did you use _metadata.file_path?"
print(f"Task 6 OK: Metadata columns added — {meta_cols}")

## Task 7: Schema Evolution — Rescued Data

Configure Auto Loader with a partial schema (only `order_id` + `customer_id`). Set `schemaEvolutionMode` to `"rescue"` so that extra columns land in `_rescued_data`.

**TODO:** Fill in the `schemaEvolutionMode` value.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

rescue_target = f"{CATALOG}.{BRONZE_SCHEMA}.orders_rescued"
rescue_checkpoint = f"{DATASET_PATH}/tmp/{CATALOG}/lab05/checkpoint_rescue"
rescue_schema = f"{DATASET_PATH}/tmp/{CATALOG}/lab05/schema_rescue"

spark.sql(f"DROP TABLE IF EXISTS {rescue_target}")
dbutils.fs.rm(rescue_checkpoint, True)
dbutils.fs.rm(rescue_schema, True)

# Partial schema — only 2 columns (provided)
partial_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
])

# TODO: Configure Auto Loader with schema rescue mode
# Set cloudFiles.schemaEvolutionMode to "rescue" so extra columns go to _rescued_data
df_rescued = (
    # YOUR CODE HERE
    ...
)

(
    df_rescued
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", rescue_checkpoint)
    .trigger(availableNow=True)
    .toTable(rescue_target)
    .awaitTermination()
)

print(f"Written to: {rescue_target}")
display(spark.table(rescue_target).limit(5))

In [0]:
# -- Validation --
rescue_cols = spark.table(rescue_target).columns
assert "_rescued_data" in rescue_cols, "Missing '_rescued_data' column — did you set schemaEvolutionMode to 'rescue'?"
rescue_count = spark.table(rescue_target).filter("_rescued_data IS NOT NULL").count()
assert rescue_count > 0, "Expected rescued data for columns not in partial schema"
print(f"Task 7 OK: {rescue_count} rows with rescued data (extra columns captured in _rescued_data)")

## Task 8: Stream-Static Join — BONUS (if time permits)

> **Note:** The demo for Stream-Static Joins is in `BONUS_05_streaming_advanced.ipynb`. Tasks 8–9 are optional — skip them if time is limited.

Join streaming data (orders) with a **static table** (customers) to enrich the stream with customer information.

A **Stream-Static Join** lets you combine a streaming DataFrame with a batch DataFrame — the static side is re-read on every micro-batch.

**TODO:** Fill in `readStream`, join columns, and `writeStream`.

In [0]:
join_target = f"{CATALOG}.{BRONZE_SCHEMA}.orders_enriched"
join_checkpoint = f"/tmp/{CATALOG}/lab05/checkpoint_enriched"

spark.sql(f"DROP TABLE IF EXISTS {join_target}")
dbutils.fs.rm(join_checkpoint, True)

# Static side: customers (provided)
customers_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers")

# TODO: Read orders as a streaming DataFrame using .format("delta").table(...)
# Use target_table (the orders table loaded in Task 1)
orders_stream = ...  # YOUR CODE HERE

# TODO: Left join orders_stream with customers_df on "customer_id"
enriched_stream = ...  # YOUR CODE HERE

# TODO: Write enriched_stream to join_target
# Use outputMode "append", checkpointLocation join_checkpoint, trigger(availableNow=True)
(
    # YOUR CODE HERE
    ...
)

print(f"Stream-static join written to: {join_target}")
display(spark.table(join_target).limit(5))

In [0]:
# -- Validation --
enriched_count = spark.table(join_target).count()
enriched_cols = spark.table(join_target).columns
assert enriched_count > 0, "No rows in enriched table"
assert "customer_id" in enriched_cols, "Missing customer_id"
assert any(c for c in enriched_cols if c not in spark.table(target_table).columns), \
    "Join didn't add any new columns — check join expression"
print(f"Task 8 OK: {enriched_count} enriched rows. Columns: {enriched_cols}")

## Task 9: Change Data Feed (CDF) for Incremental ETL — BONUS (if time permits)

> **Note:** The demo for CDF for Incremental ETL is in `BONUS_05_streaming_advanced.ipynb`.

Use **Change Data Feed** to read only the changes made to a Delta table, instead of re-reading the entire table each time.

**Scenario:** 
1. Create a CDF-enabled table with initial data
2. Make some changes (INSERT + UPDATE)
3. Read only the changes using `readChangeFeed`
4. Build an incremental ETL pipeline from the change feed

> **Exam Tip:** CDF captures `_change_type` (`insert`, `update_preimage`, `update_postimage`, `delete`), `_commit_version`, and `_commit_timestamp`.

In [0]:
cdf_source = f"{CATALOG}.{BRONZE_SCHEMA}.cdf_orders_lab"
cdf_target = f"{CATALOG}.{BRONZE_SCHEMA}.cdf_orders_incremental"

spark.sql(f"DROP TABLE IF EXISTS {cdf_source}")
spark.sql(f"DROP TABLE IF EXISTS {cdf_target}")

# TODO: Create the cdf_source table with Change Data Feed enabled
# Use TBLPROPERTIES to set delta.enableChangeDataFeed = true
spark.sql(f"""
    -- YOUR CODE HERE
""")

# Initial data (provided)
spark.sql(f"""
    INSERT INTO {cdf_source} VALUES
    (1, 101, 99.99, 'pending'),
    (2, 102, 149.50, 'pending'),
    (3, 103, 75.00, 'pending'),
    (4, 101, 200.00, 'pending'),
    (5, 104, 50.00, 'pending')
""")

initial_version = spark.sql(f"DESCRIBE HISTORY {cdf_source}").first()["version"]
print(f"CDF table created at version {initial_version} with {spark.table(cdf_source).count()} rows")

In [0]:
# Step 2: Make some changes (simulate business operations)
# Update: mark orders 1,2 as shipped
spark.sql(f"UPDATE {cdf_source} SET status = 'shipped' WHERE order_id IN (1, 2)")

# Insert: new order
spark.sql(f"INSERT INTO {cdf_source} VALUES (6, 105, 320.00, 'pending')")

print(f"Changes applied: 2 updates + 1 insert")
print(f"Current table has {spark.table(cdf_source).count()} rows")

In [0]:
# TODO: Read the Change Data Feed starting from the version AFTER the initial load
# Use the table_changes() SQL function: table_changes('table_name', start_version)
# Start from initial_version + 1 so we skip the initial INSERT

df_changes = spark.sql(f"""
    -- YOUR CODE HERE
""")

display(df_changes)

In [0]:
# TODO: Filter df_changes to keep only "insert" and "update_postimage" change types
# This removes update_preimage (old values) and deletes
# Use col("_change_type").isin(...)

df_incremental = ...  # YOUR CODE HERE

df_incremental.drop("_change_type", "_commit_version", "_commit_timestamp").write     .mode("append").saveAsTable(cdf_target)

print(f"Incremental ETL: {df_incremental.count()} rows written to {cdf_target}")
display(spark.table(cdf_target))

In [0]:
# -- Validation --
cdf_count = spark.table(cdf_target).count()
assert cdf_count == 3, f"Expected 3 rows (2 updated + 1 inserted), got {cdf_count}"
statuses = [r["status"] for r in spark.table(cdf_target).collect()]
assert "shipped" in statuses, "Should contain 'shipped' status from updates"
assert "pending" in statuses, "Should contain 'pending' status from new insert"
print(f"Task 9 OK: CDF incremental ETL — {cdf_count} change rows captured")
print("  2 update_postimage (shipped) + 1 insert (new order)")

## Cleanup

In [0]:
# Stop any active streams
for s in spark.streams.active:
    s.stop()

# Drop lab tables
for t in [target_table, al_target, 
          f"{CATALOG}.{BRONZE_SCHEMA}.orders_with_metadata",
          f"{CATALOG}.{BRONZE_SCHEMA}.orders_rescued",
          f"{CATALOG}.{BRONZE_SCHEMA}.orders_enriched",
          f"{CATALOG}.{BRONZE_SCHEMA}.cdf_orders_lab",
          f"{CATALOG}.{BRONZE_SCHEMA}.cdf_orders_incremental"]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

# Clean temp paths
for p in [checkpoint_path, schema_path,
          f"/tmp/{CATALOG}/lab05/checkpoint_metadata",
          f"/tmp/{CATALOG}/lab05/schema_metadata",
          f"/tmp/{CATALOG}/lab05/checkpoint_rescue",
          f"/tmp/{CATALOG}/lab05/schema_rescue",
          f"/tmp/{CATALOG}/lab05/checkpoint_enriched"]:
    dbutils.fs.rm(p, True)

print("Lab cleanup complete")

## Lab Complete!

You have:
- Used COPY INTO for idempotent batch loading
- Configured Auto Loader (cloudFiles) for streaming ingestion
- Used trigger(availableNow=True) for incremental processing
- Verified checkpoint-based exactly-once guarantees
- Added metadata columns (`_processing_time`, `_source_file`) to streaming writes
- Used rescued data column for schema evolution handling
- Performed a stream-static join to enrich streaming data
- Used Change Data Feed (CDF) for incremental ETL

| Feature | COPY INTO | Auto Loader | CDF |
|---------|-----------|-------------|-----|
| Format | SQL command | readStream/writeStream | readChangeFeed / table_changes() |
| Scalability | Thousands of files | Millions of files | Any Delta table |
| Schema evolution | Manual | Automatic (rescue column) | Follows source schema |
| File tracking | SQL state | Checkpoint directory | Version-based |
| Use case | Simple batch | File-based streaming | Change-based incremental |

> **Exam Tip:** Auto Loader uses `cloudFiles` format. COPY INTO is simpler but Auto Loader scales better (file notification mode for millions of files). CDF captures row-level changes and is ideal for propagating updates through Medallion layers.



← [05 — Incremental Ingestion](../demo/05_incremental_ingestion.ipynb) | **[ README](../../../README.md)** | [06 — Medallion Architecture →](../demo/06_medallion_architecture.ipynb)